# Session 15 · Homework — SOLUTIONS (teacher)

Worked solutions with commentary. All cells run top to bottom.
**Note:** the widening train-vs-test gap is *Session 16's* topic — name it here, don't
explain it away.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week","attendance_pct","sleep_hours_per_night","screen_time_hours_per_day","practice_sessions_per_week"]
X, y = df[habits], df['passed']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('train:', len(y_train), ' test:', len(y_test))

## Part 1 · Depth sweep — SOLUTION

**Train** rises with depth (0.881 → 0.922); **test** is roughly flat (0.865 → 0.881).

In [ ]:
rows = []
for d in [1, 2, 3, 4]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    rows.append({'max_depth': d, 'train_acc': round(t.score(X_train, y_train),3),
                 'test_acc': round(t.score(X_test, y_test),3)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print('Trend: training accuracy climbs as depth grows (the tree fits the training')
print('students ever more finely); test accuracy barely improves. The gap widens ->')
print('the seed of overfitting, which Session 16 pushes to the point test FALLS.')

## Part 2 · Feature importance — SOLUTION

study_hours ~**0.88** dominates; screen_time ~**0.00**.

In [ ]:
tree3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
importance = pd.Series(tree3.feature_importances_, index=habits).sort_values(ascending=False)
print(importance.round(3).to_string())
fig, ax = plt.subplots(figsize=(8,3.5))
importance[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('importance'); plt.tight_layout(); plt.show()

## Part 3 · Most-used vs ignored — SOLUTION

In [ ]:
print('most used: study_hours_per_week (~0.88)')
print('ignored  : screen_time_hours_per_day (~0.00)')
print()
print('Why ignored != unrelated: screen time is highly correlated with study hours')
print('(students who study more scroll less). Once the tree used study hours, screen')
print('time had nothing left to add -- it is REDUNDANT here, not irrelevant. Drop')
print('study hours and screen time''s importance would jump.')

## Part 4 · Counsellor note — SOLUTION (sample)

> *The model's prediction is driven almost entirely by a student's weekly study hours —
> students studying under about 8 hours a week are flagged as at risk. Attendance, sleep,
> and screen time barely move the prediction. Because the model leans so heavily on one
> habit, it's easy to explain and to act on. I'd be uncomfortable, though, if it relied on
> something like family income or postcode: those correlate with outcomes but would build
> a student's background into the prediction rather than their own changeable habits.*

**Acceptable variation:** any well-argued uncomfortable feature (a demographic proxy,
postcode, family income). Full marks require: a correct trend description (Part 1), the
redundancy caveat (Part 3), and a named uncomfortable feature with a reason (Part 4).